# Aivora AI - Stage B: financial instruction tuning

Pretraining (Stage A) ended at step 237,362 / val 4.44 / 1.88B tokens, with the
evaluation score stuck at 3/45: more pretraining had stopped helping. This
notebook fine-tunes that checkpoint to *answer questions* instead of
continuing text.

What it does:

1. builds a real instruction dataset (~50k examples) from finance-alpaca,
   Investopedia and Dolly, excluding every evaluation question;
2. fine-tunes the pretrained checkpoint with the same safeguards Stage A
   needed - gradient clipping, fp16 loss scaling, warmup, a wall-clock budget;
3. evaluates with the `### Instruction:` prompt the model is tuned on, and
   compares against the base checkpoint scored the same way.


## 1. Environment

In [ ]:
import platform, sys, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input mounted:", os.path.exists("/kaggle/input"), os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else [])


## 1b. GPU compute-capability check (before the first `import torch`)

Kaggle has been assigning this account a Tesla P100 (compute capability
6.0 / sm_60) rather than a T4, and the base image's shipped torch build
only supports compute capability 7.0+ - every real CUDA kernel launch
fails with `AcceleratorError: no kernel image is available` regardless of
`batch_size` unless an older, wider-compatibility torch build is used.

Checked via `nvidia-smi` (not `torch.cuda.get_device_capability()`)
specifically so this runs **before** `import torch` - reinstalling torch
mid-process and `importlib.reload()`-ing it is not safe (torch's C
extension re-registers native `TORCH_LIBRARY` namespaces with the
dispatcher, which crashes on a second registration). If the attached GPU
needs an older, wider-compatibility torch build, it's installed here,
before torch is ever imported for the first time.

In [ ]:
import subprocess

nvidia_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("nvidia-smi:", nvidia_smi.stdout.strip() or "(no output)", nvidia_smi.stderr.strip())

NEEDS_OLDER_TORCH = False
if nvidia_smi.returncode == 0 and nvidia_smi.stdout.strip():
    first_line = nvidia_smi.stdout.strip().splitlines()[0]
    name, _, cc_str = first_line.rpartition(",")
    try:
        compute_cap = float(cc_str.strip())
        if compute_cap < 7.0:
            NEEDS_OLDER_TORCH = True
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - below the "
                  "shipped torch build's minimum (7.0). Installing an older torch build "
                  "with wider compute-capability support before it's ever imported.")
        else:
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - "
                  "compatible with the shipped torch build, no reinstall needed.")
    except ValueError:
        print(f"Could not parse compute capability from {cc_str!r} - leaving the shipped "
              "torch build as-is and letting the GPU check cell catch any real problem.")
else:
    print("nvidia-smi query failed or returned nothing - leaving the shipped torch build "
          "as-is and letting the GPU check cell catch any real problem.")

if NEEDS_OLDER_TORCH:
    import sys
    # torch 2.7.1 (last line confirmed to still ship Pascal/sm_60 kernels)
    # + an older CUDA toolkit build (cu118) for the widest realistic
    # compute-capability coverage. This repo's attention implementation is
    # hand-written (no scaled_dot_product_attention / torch.compile
    # dependency), so an older torch build is not expected to break
    # anything model-specific.
    reinstall = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu118"],
        capture_output=True, text=True,
    )
    print(reinstall.stdout[-3000:])
    print(reinstall.stderr[-3000:])
    if reinstall.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: fallback torch==2.7.1+cu118 install failed for this "
            "compute-capability-6.0 GPU, see output above."
        )
    print("Installed torch==2.7.1+cu118 (not yet imported).")


## 2. GPU / CUDA verification (hard gate)

Raises immediately if no GPU is attached - never claims GPU training happened without this passing.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() is False. "
        "Go to Settings (right sidebar) -> Accelerator -> GPU T4 x2, "
        "save, and re-run this notebook from the top."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_COUNT = torch.cuda.device_count()
CC = torch.cuda.get_device_capability(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU AVAILABLE =", True)
print("GPU name:", GPU_NAME)
print("GPU count:", GPU_COUNT)
print("Compute capability:", CC)
print("Total VRAM (GPU 0): %.2f GB" % total_vram_gb)
print("torch version:", torch.__version__, "| CUDA build:", torch.version.cuda)


## 3. Repository transfer + integrity check

Clones the real, public repo. If this fails, Internet is probably off (Settings -> Internet -> On).

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/Ankushk-aosc/Aivora-AI.git"
REPO_DIR = "/kaggle/working/Aivora-AI"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                             capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: git clone failed (see stderr above). "
            "Most likely cause: Internet is off for this notebook "
            "(Settings -> Internet -> On), or the repo URL changed."
        )
else:
    print(f"{REPO_DIR} already present, skipping clone.")

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

required_paths = [
    "models/model.py", "training/trainer.py", "ai_platform/model_registry.py",
    "data_sources/prepare.py", "evaluation/evaluator.py", "configs/small.yaml",
    "configs/financial_poc.yaml", "inference/generator.py",
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"STATUS = BLOCKED: repo clone incomplete, missing {missing}")
print("Repo integrity check passed:", len(required_paths), "required paths present.")


## 4. Dependencies

Kaggle's base image already ships a CUDA-enabled PyTorch build tuned for the
attached GPU - reinstalling `torch` over it risks silently downgrading to a
CPU or mismatched-CUDA wheel. This only installs the *other* requirements,
and reuses `torch.cuda.is_available()` from Cell 2 to confirm nothing broke
it afterward.

In [ ]:
import subprocess, sys

with open("requirements.txt") as f:
    reqs = [line.strip() for line in f if line.strip() and not line.startswith("#")]

# torch is already provided by the Kaggle GPU image - skip it here.
reqs_to_install = [r for r in reqs if not r.lower().startswith("torch")]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + reqs_to_install,
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError("STATUS = BLOCKED: pip install failed, see output above.")

check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print(check.stdout)
print(check.stderr)

# NOTE: deliberately do NOT importlib.reload(torch) here - torch's C
# extension init is not reload-safe (it re-registers native TORCH_LIBRARY
# namespaces with the dispatcher, which crashes on a second registration).
# Since torch itself was excluded from the install above, the already-
# imported torch module from Cell 2 is still valid and doesn't need reloading.
if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() became False after "
        "installing requirements.txt - something in the dependency list "
        "pulled in a CPU-only torch build. Check requirements.txt for a "
        "torch pin and remove it."
    )
print("Dependencies installed; CUDA still available after install.")


## 5. Base checkpoint (Stage A output)

Takes the highest-step checkpoint from the attached dataset. Stage B starts
from pretrained weights; it never trains from scratch.

In [ ]:
import glob as _glob
import re as _re

candidates = _glob.glob("/kaggle/input/**/checkpoint_*.pt", recursive=True)
if not candidates:
    raise RuntimeError(
        "STATUS = BLOCKED: no checkpoint_*.pt under /kaggle/input. Attach the "
        "aivora-ai-financial-poc-checkpoint-latest dataset."
    )


def _step(p):
    m = _re.search(r"checkpoint_(\d+)\.pt$", p)
    return int(m.group(1)) if m else -1


BASE_CHECKPOINT = max(candidates, key=_step)
MIN_BASE_STEP = 237362
if _step(BASE_CHECKPOINT) < MIN_BASE_STEP:
    raise RuntimeError(
        f"STATUS = BLOCKED: newest checkpoint is {BASE_CHECKPOINT}, older than the "
        f"final pretrained step {MIN_BASE_STEP}."
    )
print(f"Base checkpoint: {BASE_CHECKPOINT} (step {_step(BASE_CHECKPOINT)})")

## 6. Build the instruction dataset

`data_sources/build_instruction_dataset.py` streams from Hugging Face and
drops rows that are too short, duplicated, or that match an evaluation
question. The dataset that shipped in the repo had 800 rows (the first 800 of
finance-alpaca), which is far too few to teach answering.

In [ ]:
from data_sources.build_instruction_dataset import build

stats = build()
INSTRUCTION_DATA = stats["path"]
print("")
print(f"Instruction examples: {stats['total']:,}")
print(f"By source: {stats['by_source']}")
print(f"Dropped: {stats['dropped']}")
assert stats["dropped"]["leaked"] == 0 or True  # leaked rows are excluded, not fatal
assert stats["total"] > 10000, "too few instruction examples to fine-tune on"

## 7. Baseline: score the BASE checkpoint first

Without this, any score after tuning has nothing to compare against. The base
model is scored with the `Question:/Answer:` prompt it was pretrained on.

In [ ]:
from evaluation import evaluate_model, print_report
from inference import load_model_for_inference

base_model, base_config = load_model_for_inference(BASE_CHECKPOINT, device="cuda")
base_results = evaluate_model(base_model, device="cuda", max_new_tokens=48,
                              verbose=False, prompt_style="qa")
print_report(base_results)
BASE_SCORE = base_results["overall"]["accuracy"]
del base_model
import torch
torch.cuda.empty_cache()

## 8. Fine-tune

Settings and why:

| setting | value | reason |
| --- | --- | --- |
| learning rate | `2e-5` | below the `3e-5` pretraining peak; the old default `1e-4` risks undoing pretraining |
| grad clipping | `1.0` | a local test hit gradient norms of 20.7 - unclipped, that would wreck the weights |
| GradScaler | on for fp16 | T4s run fp16; without scaling small gradients underflow to zero |
| effective batch | 8 x 4 | fits the T4s at seq_len 512 |
| budget | 5 h | Kaggle stops a session at 12 h; finishing normally is what persists outputs |

In [ ]:
import os

from training.instruction_trainer import train_instruction

SFT_DIR = "/kaggle/working/checkpoints/instruction"
os.makedirs(SFT_DIR, exist_ok=True)

import torch
n_gpus = max(1, torch.cuda.device_count())
BATCH = 8 * n_gpus          # DataParallel splits this across GPUs
ACCUM = max(1, 4 // n_gpus)  # effective batch stays 32

model, config, sft_ckpt = train_instruction(
    BASE_CHECKPOINT,
    data_path=INSTRUCTION_DATA,
    max_steps=6000,
    batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    seq_len=512,
    learning_rate=2e-5,
    min_lr=2e-6,
    warmup_steps=200,
    eval_interval=250,
    eval_iters=20,
    checkpoints_dir=SFT_DIR,
    max_train_seconds=5 * 3600,
)
print(f"\nStage B checkpoint: {sft_ckpt}")

## 9. Score the tuned model and compare

The tuned model is prompted with the `### Instruction:` template it was
trained on; scoring it with the base prompt would measure the wrong thing.

In [ ]:
sft_results = evaluate_model(model, device="cuda", max_new_tokens=48,
                             verbose=True, prompt_style="instruction")
print_report(sft_results)
SFT_SCORE = sft_results["overall"]["accuracy"]
print("")
print(f"base checkpoint : {BASE_SCORE}")
print(f"instruction-tuned: {SFT_SCORE}")

## 10. Sample answers

The numbers above only count keyword matches; read these to judge the answers
yourself.

In [ ]:
from training.instruction_dataset import format_prompt
from evaluation.evaluator import generate_answer

PROMPTS = [
    "What is EBITDA?",
    "What is working capital?",
    "Explain gross margin in one sentence.",
    "What does a balance sheet show?",
    "Why do companies issue bonds?",
]
for q in PROMPTS:
    out = generate_answer(model, format_prompt({"instruction": q, "input": ""}),
                          max_new_tokens=64, device="cuda")
    print(f"\nQ: {q}\nA: {out.strip()}")

## 11. Export

Registers the tuned checkpoint and verifies its checksum, the same way the
pretraining notebook does.

In [ ]:
import json

from ai_platform.model_registry import register_checkpoint, verify_integrity

entry = register_checkpoint(sft_ckpt, stage="instruction", set_active=True)
print("Registered:", entry)
print("Integrity :", verify_integrity(entry["version"]))

with open("/kaggle/working/instruction_manifest.json", "w") as f:
    json.dump({"base_checkpoint": BASE_CHECKPOINT, "sft_checkpoint": sft_ckpt,
               "base_score": BASE_SCORE, "sft_score": SFT_SCORE,
               "instruction_data": stats, "registry_entry": entry}, f, indent=2)
print("Wrote instruction_manifest.json")